# Eddy flux 

This notebook shows how to plot eddy flux data with `fluxy`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
from pathlib import Path
logging.basicConfig(level=logging.INFO)
logging.getLogger('matplotlib').setLevel(logging.WARNING)

from fluxy.config import set_print_settings
from fluxy.io import read_config_files
from fluxy.test_utils import data_dir
from fluxy.plots.mf_timeseries import plot_timeseries, plot_sites_timeseries
from fluxy.plots.ec_flux.sectorial_stack import plot_stacked
from fluxy.operators.sectors import group_sectors
from fluxy.operators.ecflux import  filter_ecflux 


import matplotlib.pyplot as plt
import xarray as xr
plt.style.use('default')


species = 'CO2'

data_dir = data_dir.parent / 'my_data' / "eddy_pro" 
config_dir = Path.cwd() / 'configs'

config_dict = read_config_files(config_dir)
config_dict.keys()


In [ ]:
from fluxy.io import read_model_output


dss_raw = read_model_output(
    data_dir=data_dir,
    file_type="eddy_flux",
    species=species, 
    models=['EDDY_HARDAU', "EDDY_HARDAU_STORAGE_2LAYERS"]

)
list(map(display, dss_raw.values()))

In [ ]:

fig = plot_timeseries(
    dss_raw,
    species='CO2',
    site='HARDAU',
    include={
        #"ecflux_observed": "flux_observed_random_error",
        "ecflux_observed": None,
        #"ecflux_observed_storage": None,
    },
    n_bins= 20,

)

In [ ]:

dss = filter_ecflux(dss_raw)


In [ ]:
fig = plot_timeseries(
    dss,
    species='CO2',
    site='HARDAU',
    include={
        #"ecflux_observed": "flux_observed_random_error",
        "ecflux_observed": None,
        #"ecflux_observed_storage": None,
    },
    n_bins= 100,

)

## Correlation plots

In [ ]:
from fluxy.plots.correlation import plot_correlation
from fluxy.operators.flux_align_dataset import align_time


fig = plot_correlation(
    dss,
    variable="ecflux_observed_storage",
    style='density'
)

In [ ]:
from fluxy.operators.flux_align_dataset import align_time

try:
    fig = plot_correlation(
        {'EDDY_HARDAU': dss['EDDY_HARDAU']},
        variable=["ecflux_observed_storage", "ecflux_observed"],
        style='density',
        oppose='variables',
        lims=(-40, 40)
    )
except Exception as e:
    print(e)



## Sectorial plots

In [ ]:


dss = group_sectors(dss,sectors_config=config_dict['sectors'],)

In [ ]:
dss.keys()

In [ ]:

for season in ['DJF', 'MAM', 'JJA', 'SON']:

    fig, ax = plot_stacked(
        dss['EDDY_HARDAU_STORAGE_2LAYERS'],
        sectors_config=config_dict['sectors'],
        season=season,
        area=True
        
    )

In [ ]:



fig, ax = plot_stacked(
    dss['EDDY_HARDAU'],
    sectors_config=config_dict['sectors'],
    
)